# CellChat on Spatial Data: Step-by-Step Tutorial

Reproducible code for the blog post at [spatiabio.com](https://www.spatiabio.com).

This notebook uses the stxBrain Visium dataset (Seurat) to run CellChat in spatial mode.

## R code

This analysis is in R. Run in RStudio or via `rmarkdown::render()`.

In [ ]:
# This cell shows the R code as reference
# Run in R, not Python

r_code = '''
library(Seurat)
library(SeuratData)
library(CellChat)

# Load data
brain <- LoadData("stxBrain", type = "anterior1")
brain <- SCTransform(brain, assay = "Spatial", verbose = FALSE)
brain <- RunPCA(brain, assay = "SCT", verbose = FALSE)
brain <- FindNeighbors(brain, reduction = "pca", dims = 1:30, verbose = FALSE)
brain <- FindClusters(brain, verbose = FALSE)

# Prepare CellChat input
data.input <- GetAssayData(brain, layer = "data", assay = "SCT")  # Seurat v5
meta <- data.frame(labels = factor(paste0("C", Idents(brain))),
                   row.names = names(Idents(brain)))
spatial.locs <- as.matrix(GetTissueCoordinates(brain)[, c("x","y")])

cellchat <- createCellChat(object = data.input, meta = meta, group.by = "labels",
                           datatype = "spatial", coordinates = spatial.locs,
                           spatial.factors = data.frame(ratio = 1, tol = 5))

# Run CellChat
cellchatDB <- CellChatDB.mouse
cellchat <- subsetData(cellchat)
cellchat <- identifyOverExpressedGenes(cellchat)
cellchat <- identifyOverExpressedInteractions(cellchat)
cellchat <- computeCommunProb(cellchat, type = "truncatedMean", trim = 0.1,
                              distance.use = TRUE, interaction.range = 250,
                              scale.distance = 0.01, contact.dependent = TRUE,
                              contact.range = 100)
cellchat <- filterCommunication(cellchat, min.cells = 10)
cellchat <- aggregateNet(cellchat)
'''
print(r_code)

## Common errors and fixes

**Error 1**: `GetAssayData()` with `slot` is defunct
- Fix: use `layer=` instead of `slot=` (Seurat v5)

**Error 2**: Spatial coordinates not found
- Fix: use `GetTissueCoordinates(brain)[, c('x','y')]`

**Error 3**: `contact.range` parameter missing
- Fix: update CellChat to >= 2.1.2